# LEMoE Query Distiller - Token Classification Training (DeBERTa)

This notebook automates the fine-tuning process of the microsoft/mdeberta-v3-base model for the query distillation task using token classification (Binary NER: 0 = DISCARD, 1 = KEEP). 

The dataset is loaded directly from a **JSONL** (JSON Lines) formatted file.

### 1. Dependencies Installation
Install the required tools for fine-tuning, token classification, and ONNX exportation.

In [3]:
!pip install transformers datasets accelerate sentencepiece optimum[onnxruntime] evaluate seqeval

Defaulting to user installation because normal site-packages is not writeable


### 2. Creation of a sample JSONL file
To ensure the notebook is immediately runnable, we create a sample dataset.jsonl file with the required structure: a list of words (tokens) and a list of corresponding numerical tags (ner_tags).

In [5]:
import json

# Sample data to simulate the input file
ejemplos = [
    {
        "tokens": ["búscame", "la", "factura", "de", "iberdrola", "por", "favor"],
        "ner_tags": [0, 0, 1, 0, 1, 0, 0]
    },
    {
        "tokens": ["necesito", "ver", "el", "contrato", "de", "alquiler", "de", "la", "oficina"],
        "ner_tags": [0, 0, 0, 1, 0, 1, 0, 0, 1]
    },
    {
        "tokens": ["muestrame", "las", "auditorias", "del", "core", "de", "lemoe"],
        "ner_tags": [0, 0, 1, 0, 1, 0, 1]
    },
    {
        "tokens": ["quiero", "el", "recibo", "de", "la", "luz"],
        "ner_tags": [0, 0, 1, 0, 0, 1]
    },
    {
        "tokens": ["búscame", "las", "aoditorias", "del", "sistema"],
        "ner_tags": [0, 0, 1, 0, 1]
    }
]

# Write the JSONL file
with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for ejemplo in ejemplos:
        f.write(json.dumps(ejemplo, ensure_ascii=False) + "\n")

print("File 'dataset.jsonl' successfully created.")

Archivo 'dataset.jsonl' creado correctamente.


### 3. Loading the JSONL Dataset
We use the Hugging Face datasets library to parse the .jsonl file efficiently.

In [6]:
from datasets import load_dataset

# Load the local JSONL file
raw_dataset = load_dataset("json", data_files="data.jsonl", split="train")

# Mapping of identifiers and hierarchical labels
id2label = {0: "DISCARD", 1: "KEEP"}
label2id = {"DISCARD": 0, "KEEP": 1}

print("Loaded dataset structure:")
print(raw_dataset)

Estructura del dataset cargado:
Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 1043
})


### 4. Tokenization and Label Alignment
Because WordPiece or SentencePiece tokenizers can split a word into multiple sub-tokens, it's necessary to align the original ner_tags array with the new tokens generated by the model. We apply the -100 label to secondary sub-tokens to ignore them during loss calculation.

In [7]:
from transformers import AutoTokenizer

model_name = "microsoft/mdeberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels = []
    
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special mapping tokens (like padding/CLS) receive -100
            if word_idx is None:
                label_ids.append(-100)
            # Assign the original label to the first sub-token of the word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # The remaining sub-tokens of the same word are ignored with -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Process the complete dataset
tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=True)

/home/jrodriiguezg/.local/lib/python3.14/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


### 5. Model Configuration and Training
We initialize the token classification architecture and set hyperparameters optimized for small datasets.

In [8]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

# Base architecture loading with final classification layer (2 classes)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=2, 
    id2label=id2label, 
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Optimized configuration for a small dataset
training_args = TrainingArguments(
    output_dir="./deberta-query-distiller",
    learning_rate=3e-5,
    per_device_train_batch_size=16, # Increased to 16 to leverage VRAM
    num_train_epochs=10,            # 10 epochs are sufficient for this dataset size
    weight_decay=0.01,
    logging_steps=10,               # To monitor loss reduction more frequently
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Execute fine-tuning
trainer.train()

# Save intermediate weights in PyTorch format
trainer.save_model("./deberta_finetuned")

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_12724/825842318.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
10,0.537000
20,0.141500
30,0.126800
40,0.070800
50,0.018100
60,0.027200
70,0.069300
80,0.009400
90,0.047800
100,0.043200


### 6. Export to ONNX Format
We transform the dynamic PyTorch graph to an optimized static format for fast CPU executions using optimum.

In [ ]:
from optimum.onnxruntime import ORTModelForTokenClassification
from transformers import AutoTokenizer
import os

onnx_model_path = "./deberta_onnx"
model_source = "./deberta-query-distiller"

# Load tokenizer with regex fix to avoid warnings
tokenizer = AutoTokenizer.from_pretrained(
    model_source, 
    fix_mistral_regex=True
)

# Direct model export to ONNX format
ort_model = ORTModelForTokenClassification.from_pretrained(
    model_source, 
    export=True
)

# Persist the ONNX model and its tokenizer
ort_model.save_pretrained(onnx_model_path)
tokenizer.save_pretrained(onnx_model_path)

print(f"ONNX Model successfully exported at: {os.path.abspath(onnx_model_path)}")

/home/jrodriiguezg/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
The tokenizer you are loading from './deberta_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from './deberta_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer 

Modelo ONNX exportado exitosamente en: /home/jrodriiguezg/Documentos/Proyectos/IA/Lemoe/LEMoE - Light Easy Mix Of Experts/private/train/cleaner/esp/deberta_onnx


### 7. Production Inference Script (ONNX Runtime)
This final block simulates production environment behavior. It only requires onnxruntime and transformers to process inputs in milliseconds.

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
import time

# 1. Path to folder containing 'model.safetensors' and tokenizer files
model_path = "./deberta_finetuned" 

# 2. Load tokenizer and native model
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Force GPU usage for ultra-fast inference
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = AutoModelForTokenClassification.from_pretrained(model_path).to(device)

# Set model to evaluation mode (disables dropout layers, etc.)
model.eval()

def clean_query_safetensors(query: str) -> str:
    # Generate PyTorch tensors and send them to GPU
    inputs = tokenizer(query, return_tensors="pt", truncation=True).to(device)
    
    # Pure inference without gradient calculation (saves memory and time)
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Extract logits and apply argmax to get the class (0 or 1)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()
    
    word_ids = inputs.word_ids()
    word_preds = {}
    
    # Map the prediction of the first sub-token to the full word
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_preds:
            word_preds[word_id] = predictions[idx]
            
    # Reconstruct final phrase by slicing exact characters
    keep_words = []
    for word_id, pred in word_preds.items():
        if pred == 1:  # Etiqueta 1 corresponde a KEEP
            start_char, end_char = inputs.word_to_chars(word_id)
            keep_words.append(query[start_char:end_char])
            
    return " ".join(keep_words)

# --- Performance Test ---
test_query = "¿Podrías buscarme por favor la última factura de Iberdrola del mes pasado?"

# VRAM warmup
_ = clean_query_safetensors("warmup")

start = time.perf_counter()
result = clean_query_safetensors(test_query)
end = time.perf_counter()

print(f"Original input: '{test_query}'")
print(f"Distilled output: '{result}'")
print(f"Inference time: {(end - start) * 1000:.2f} ms")

The tokenizer you are loading from './deberta_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Entrada original: '¿Podrías buscarme por favor la última factura de Iberdrola del mes pasado?'
Salida destilada: ' factura  Iberdrola  mes  pasado?'
Tiempo de inferencia: 26.00 ms
